<a href="https://colab.research.google.com/github/gabriel-tfg/TFM_RAG_Pipeline/blob/main/PruebasTFM7_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install transformers

In [ ]:
pip install faiss-cpu

1) Query configuration

In [ ]:
import requests
import xml.etree.ElementTree as ET
import json
import re
from collections import defaultdict

QUERY_1 = """
"meta-analysis"[Publication Type] AND ("Menopause"[MeSH]) AND
("Quality of Life"[MeSH]
OR "Sleep Wake Disorders"[MeSH]
OR "Stress, Psychological"[MeSH]
OR "Fatigue"[MeSH]
OR "Anxiety"[MeSH]
OR "Cognition Disorders"[MeSH]
OR "Cognitive Dysfunction"[MeSH]
OR "Executive Function"[MeSH]
OR "Memory"[MeSH]
OR cognit*[tiab]
OR fatigue[tiab])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_2 = """
("meta-analysis"[Publication Type] AND "menopause/psychology"[MeSH Terms])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_3 = """
("meta-analysis"[Publication Type] AND "menopause/physiology"[MeSH Terms])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

ACTIVE_QUERY = QUERY_1   # cambiar a QUERY_2 o QUERY_3 para pruebas
RETMAX = 100

2) Pubmed search helpers

In [ ]:
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH_URL  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
ELINK_URL   = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"

def pubmed_search(query, retmax=20):
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json"
    }
    data = requests.get(ESEARCH_URL, params=params).json()
    return data["esearchresult"]["idlist"]

def pubmed_fetch_xml(pmids):
    if not pmids:
        return None
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml"
    }
    xml_text = requests.get(EFETCH_URL, params=params).content
    return ET.fromstring(xml_text)

def parse_pubmed_articles(root):
    docs = []
    if root is None:
        return docs

    for i, article in enumerate(root.findall(".//PubmedArticle")):
        pmid_elem = article.find(".//PMID")
        title_elem = article.find(".//ArticleTitle")
        abstract_elems = article.findall(".//Abstract/AbstractText")

        pmid = pmid_elem.text.strip() if pmid_elem is not None and pmid_elem.text else None
        if title_elem is None:
            continue

        title = "".join(title_elem.itertext()).strip()
        abstract = " ".join("".join(a.itertext()).strip() for a in abstract_elems) if abstract_elems else ""

        text = f"{title}\n\n{abstract}".strip()

        docs.append({
            "id": f"meta_{pmid}" if pmid else f"meta_{i}",
            "pmid": pmid,
            "type": "meta-analysis",
            "title": title,
            "text": text
        })

    return docs

# Buscar metaanálisis
meta_pmids = pubmed_search(ACTIVE_QUERY, retmax=RETMAX)
print("Meta-analysis PMIDs:", len(meta_pmids))

meta_root = pubmed_fetch_xml(meta_pmids)
meta_docs = parse_pubmed_articles(meta_root)

print("Meta-analysis docs:", len(meta_docs))
print(meta_docs[0]["title"] if meta_docs else "No docs")

Meta-analysis PMIDs: 32
Meta-analysis docs: 32
Can massage ameliorate menopausal and postmenopausal symptoms in women? A systematic review and meta-analysis.
Meta-analysis PMIDs: 32
Meta-analysis docs: 32
Can massage ameliorate menopausal and postmenopausal symptoms in women? A systematic review and meta-analysis.


3) Link PubMed -> PMC

In [ ]:
import requests
import time

ID_CONVERTER_URL = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"

def get_pmcid_for_pmid(pmid, tool="rag_pipeline", email="tu_email@ejemplo.com"):
    params = {
        "ids": str(pmid),
        "format": "json",
        "tool": tool,
        "email": email
    }

    try:
        r = requests.get(ID_CONVERTER_URL, params=params, timeout=20)
        r.raise_for_status()

        # Ver qué tipo de contenido devuelve realmente
        content_type = r.headers.get("Content-Type", "")

        if "json" not in content_type.lower():
            print(f"[Aviso] PMID {pmid}: la respuesta no es JSON.")
            print("Content-Type:", content_type)
            print("Respuesta:", r.text[:300])
            return None

        if not r.text.strip():
            print(f"[Aviso] PMID {pmid}: respuesta vacía.")
            return None

        data = r.json()
        records = data.get("records", [])
        if not records:
            print(f"[Aviso] PMID {pmid}: sin records.")
            return None

        pmcid = records[0].get("pmcid")
        return pmcid

    except requests.exceptions.RequestException as e:
        print(f"[Error HTTP] PMID {pmid}: {e}")
        return None
    except ValueError as e:
        print(f"[Error JSON] PMID {pmid}: {e}")
        print("Respuesta recibida:", r.text[:300])
        return None

meta_to_pmc = {}
for d in meta_docs:
    if d.get("pmid"):
        meta_to_pmc[d["pmid"]] = get_pmcid_for_pmid(d["pmid"])
        time.sleep(0.3)

print(meta_to_pmc)

{'41903824': None, '41531400': 'PMC12835450', '41448220': None, '41401249': None, '41401244': None, '41341454': 'PMC12669000', '41161257': 'PMC12597306', '41129871': None, '41066270': None, '40971657': None, '40742785': None, '40718787': 'PMC12296567', '40575963': None, '40288155': None, '40231842': 'PMC12599504', '40224208': 'PMC11992344', '40105038': None, '39987726': None, '39919442': None, '39856668': 'PMC11762881', '39145901': None, '38977980': 'PMC11229230', '38619017': None, '38563867': None, '38489855': None, '37960761': 'PMC10637479', '37945913': None, '37697662': 'PMC10594314', '37477236': None, '36147572': 'PMC9486389', '34758929': None, '33880736': 'PMC8595144'}
{'41903824': None, '41531400': 'PMC12835450', '41448220': None, '41401249': None, '41401244': None, '41341454': 'PMC12669000', '41161257': 'PMC12597306', '41129871': None, '41066270': None, '40971657': None, '40742785': None, '40718787': 'PMC12296567', '40575963': None, '40288155': None, '40231842': 'PMC12599504', '

4) PMC XML retrieval + reference extraction

In [ ]:
from collections import defaultdict
import requests
import xml.etree.ElementTree as ET
import time
import re

PMC_OAI_URL = "https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/"

def fetch_pmc_xml_oai(pmcid):
    if not pmcid:
        return None

    pmc_num = pmcid.replace("PMC", "").strip()

    params = {
        "verb": "GetRecord",
        "identifier": f"oai:pubmedcentral.nih.gov:{pmc_num}",
        "metadataPrefix": "pmc"
    }

    try:
        r = requests.get(PMC_OAI_URL, params=params, timeout=30)
        r.raise_for_status()

        if not r.text.strip():
            print(f"[Aviso] {pmcid}: respuesta vacía")
            return None

        return r.text

    except requests.exceptions.RequestException as e:
        print(f"[Error HTTP] {pmcid}: {e}")
        return None

def extract_article_xml_from_oai(oai_xml_text):
    if not oai_xml_text:
        return None

    try:
        root = ET.fromstring(oai_xml_text)
    except Exception as e:
        print(f"[Error XML OAI] {e}")
        return None

    ns = {
        "oai": "http://www.openarchives.org/OAI/2.0/"
    }

    metadata = root.find(".//oai:metadata", ns)
    if metadata is None or len(metadata) == 0:
        print("[Aviso] No se encontró bloque <metadata> en la respuesta OAI")
        return None

    article_elem = list(metadata)[0]

    try:
        return ET.tostring(article_elem, encoding="unicode")
    except Exception as e:
        print(f"[Error serializando article XML] {e}")
        return None

def extract_references_from_pmc_xml(article_xml_text):
    refs = []
    if not article_xml_text:
        return refs

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando article XML] {e}")
        return refs

    for i, ref in enumerate(root.findall(".//{*}ref")):
        ref_text = " ".join(t.strip() for t in ref.itertext() if t and t.strip())

        pmid = None
        doi = None
        year = None
        first_author = None

        ref_id = ref.attrib.get("id", f"ref_{i}")

        for pubid in ref.findall(".//{*}pub-id"):
            id_type = pubid.attrib.get("pub-id-type", "").lower()
            if pubid.text:
                if id_type == "pmid" and pmid is None:
                    pmid = pubid.text.strip()
                elif id_type == "doi" and doi is None:
                    doi = pubid.text.strip()

        year_match = re.search(r"\b(19|20)\d{2}\b", ref_text)
        if year_match:
            year = year_match.group(0)

        surname_elem = ref.find(".//{*}surname")
        if surname_elem is not None and surname_elem.text:
            first_author = surname_elem.text.strip().lower()

        refs.append({
            "ref_id": ref_id,
            "pmid": pmid,
            "doi": doi,
            "year": year,
            "first_author": first_author,
            "ref_text": ref_text
        })

    return refs

4.1) Included study detection (heuristic)

In [ ]:
import xml.etree.ElementTree as ET

def extract_candidate_sections(article_xml_text):
    sections = []
    if not article_xml_text:
        return sections

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando XML para secciones] {e}")
        return sections

    for sec in root.findall(".//{*}sec"):
        title_elem = sec.find("./{*}title")
        title = ""
        if title_elem is not None:
            title = " ".join(t.strip() for t in title_elem.itertext() if t and t.strip()).lower()

        sec_text = " ".join(t.strip() for t in sec.itertext() if t and t.strip())
        sections.append({
            "title": title,
            "text": sec_text.lower()
        })

    return sections

def extract_candidate_tables(article_xml_text):
    tables = []
    if not article_xml_text:
        return tables

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando XML para tablas] {e}")
        return tables

    for tbl in root.findall(".//{*}table-wrap"):
        label_elem = tbl.find(".//{*}label")
        caption_elem = tbl.find(".//{*}caption")

        label = " ".join(label_elem.itertext()).strip().lower() if label_elem is not None else ""
        caption = " ".join(caption_elem.itertext()).strip().lower() if caption_elem is not None else ""
        text = " ".join(t.strip() for t in tbl.itertext() if t and t.strip()).lower()

        tables.append({
            "label": label,
            "caption": caption,
            "text": text
        })

    return tables

def score_reference_as_included(ref, sections, tables):
    score = 0
    evidence = []

    first_author = ref.get("first_author")
    year = ref.get("year")

    if ref.get("pmid"):
        score += 2
        evidence.append("has_pmid")

    relevant_titles = [
        "included studies",
        "characteristics of included studies",
        "study characteristics",
        "included study",
        "results",
        "studies included",
        "study selection"
    ]

    for sec in sections:
        sec_title = sec["title"]
        sec_text = sec["text"]

        title_is_relevant = any(key in sec_title for key in relevant_titles)

        if first_author and year and first_author in sec_text and year in sec_text:
            if title_is_relevant:
                score += 5
                evidence.append(f"section:{sec_title}")
            else:
                score += 2
                evidence.append("author_year_in_section")

    table_keywords = [
        "included",
        "characteristics",
        "study",
        "studies"
    ]

    for tbl in tables:
        tbl_meta = f"{tbl['label']} {tbl['caption']}"
        tbl_text = tbl["text"]
        table_is_relevant = any(k in tbl_meta for k in table_keywords)

        if first_author and year and first_author in tbl_text and year in tbl_text:
            if table_is_relevant:
                score += 6
                evidence.append(f"table:{tbl_meta[:80]}")
            else:
                score += 3
                evidence.append("author_year_in_table")

    return score, evidence

def extract_included_study_candidates(article_xml_text, min_score=5):
    refs = extract_references_from_pmc_xml(article_xml_text)
    sections = extract_candidate_sections(article_xml_text)
    tables = extract_candidate_tables(article_xml_text)

    included = []
    all_scored = []

    for ref in refs:
        score, evidence = score_reference_as_included(ref, sections, tables)

        item = {
            **ref,
            "score": score,
            "evidence": evidence
        }
        all_scored.append(item)

        if score >= min_score:
            included.append(item)

    included = sorted(included, key=lambda x: x["score"], reverse=True)
    all_scored = sorted(all_scored, key=lambda x: x["score"], reverse=True)

    return included, all_scored

4.2) Test included-study detection + build mappings

In [ ]:
from collections import defaultdict
import time

test_pmcid = "PMC12835450"

oai_xml = fetch_pmc_xml_oai(test_pmcid)
print(oai_xml[:500] if oai_xml else "No OAI XML")

article_xml = extract_article_xml_from_oai(oai_xml)
print(article_xml[:500] if article_xml else "No article XML")

refs = extract_references_from_pmc_xml(article_xml)
print("N refs:", len(refs))
print("First refs:", refs[:3])

included, all_scored = extract_included_study_candidates(article_xml, min_score=5)
print("Included candidates:", len(included))
print("Top included candidates:", included[:5])

meta_included_candidates = defaultdict(list)
meta_all_scored_refs = defaultdict(list)

for d in meta_docs:
    pmid = d.get("pmid")
    pmcid = meta_to_pmc.get(pmid)

    if pmcid:
        oai_xml = fetch_pmc_xml_oai(pmcid)
        article_xml = extract_article_xml_from_oai(oai_xml)

        included, all_scored = extract_included_study_candidates(article_xml, min_score=5)

        meta_included_candidates[pmid] = included
        meta_all_scored_refs[pmid] = all_scored

        print(f"Meta PMID {pmid}: included candidates = {len(included)} / all refs = {len(all_scored)}")
        time.sleep(0.34)

print("Example meta PMID:", next(iter(meta_included_candidates)) if meta_included_candidates else "No meta")




<OAI-PMH xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/ http://www.openarchives.org/OAI/2.0/OAI-PMH.xsd">
    <responseDate>2026-05-17T18:29:41Z</responseDate>
    <request verb="GetRecord" identifier="oai:pubmedcentral.nih.gov:12835450" metadataPrefix="pmc">https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/</request>
    
    <GetRecord>
        <record>
    <header>
    <identifier
<ns0:article xmlns:ns0="https://jats.nlm.nih.gov/ns/archiving/1.4/" xmlns:ns2="http://www.niso.org/schemas/ali/1.0/" xmlns:ns3="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="https://jats.nlm.nih.gov/ns/archiving/1.4/ https://jats.nlm.nih.gov/archiving/1.4/xsd/JATS-archivearticle1-4.xsd" xml:lang="en" article-type="research-article" dtd-version="1.4"><ns0:processing-meta base-tagset="archiving" mathml-version="3.0" table-model="xhtml" tag

5) Collect cited PMIDs from references

In [ ]:
included_pmids = set()

for meta_pmid, refs in meta_included_candidates.items():
    for ref in refs:
        if ref.get("pmid"):
            included_pmids.add(str(ref["pmid"]))

included_pmids = sorted(included_pmids)

print("Included-study candidate PMIDs found:", len(included_pmids))
print(included_pmids[:10])

Included-study candidate PMIDs found: 114
['10183310', '11457767', '14412840', '14706761', '1484915', '15511602', '16024761', '16522699', '16568338', '16737531']
Included-study candidate PMIDs found: 114
['10183310', '11457767', '14412840', '14706761', '1484915', '15511602', '16024761', '16522699', '16568338', '16737531']


5.2) Save included PMIDs + prepare new PMIDs

In [ ]:
import json

# 1) Guardar todos los PMIDs incluidos (heurísticos)
with open("included_pmids.json", "w", encoding="utf-8") as f:
    json.dump(included_pmids, f, ensure_ascii=False, indent=2)

# 2) PMIDs semilla (metaanálisis)
seed_pmids = {
    str(d["pmid"])
    for d in meta_docs
    if d.get("pmid")
}

# 3) Filtrar: quitar los metaanálisis
new_included_pmids = [
    str(pmid) for pmid in included_pmids
    if str(pmid) not in seed_pmids
]

# 4) Ordenar
new_included_pmids = sorted(set(new_included_pmids))

# 5) Guardar
with open("new_included_pmids.json", "w", encoding="utf-8") as f:
    json.dump(new_included_pmids, f, ensure_ascii=False, indent=2)

print("Seed PMIDs:", len(seed_pmids))
print("All included PMIDs:", len(included_pmids))
print("New included PMIDs:", len(new_included_pmids))
print(new_included_pmids[:10])

Seed PMIDs: 32
All included PMIDs: 114
New included PMIDs: 114
['10183310', '11457767', '14412840', '14706761', '1484915', '15511602', '16024761', '16522699', '16568338', '16737531']
Seed PMIDs: 32
All included PMIDs: 114
New included PMIDs: 114
['10183310', '11457767', '14412840', '14706761', '1484915', '15511602', '16024761', '16522699', '16568338', '16737531']


6) Fetch cited papers

In [ ]:
included_root = pubmed_fetch_xml(new_included_pmids[:200])
included_docs_raw = parse_pubmed_articles(included_root)

included_docs = []
for d in included_docs_raw:
    included_docs.append({
        "id": f"included_{d['pmid']}" if d.get("pmid") else d.get("id"),
        "pmid": d.get("pmid"),
        "type": "included-study-candidate",
        "title": d.get("title"),
        "text": d.get("text")
    })

print("Included-study candidate docs:", len(included_docs))
print(included_docs[0]["title"] if included_docs else "No docs")

Included-study candidate docs: 114
The MOS 36-item short form health survey. A conceptual analysis.
Included-study candidate docs: 114
The MOS 36-item short form health survey. A conceptual analysis.


7) Final corpus + repository files

In [ ]:
import json

seen = set()
docs = []

for d in meta_docs + included_docs:
    pmid = d.get("pmid")
    if pmid and pmid not in seen:
        docs.append(d)
        seen.add(pmid)

print("TOTAL DOCS IN CORPUS:", len(docs))

# guardar corpus
with open("corpus_meta_plus_included.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

# guardar mapping meta → included candidates
with open("meta_included_map.json", "w", encoding="utf-8") as f:
    json.dump(dict(meta_included_candidates), f, ensure_ascii=False, indent=2)

print("Saved corpus_meta_plus_included.json")
print("Saved meta_included_map.json")

TOTAL DOCS IN CORPUS: 146
Saved corpus_meta_plus_included.json
Saved meta_included_map.json
TOTAL DOCS IN CORPUS: 146
Saved corpus_meta_plus_included.json
Saved meta_included_map.json


8) Chunking

In [ ]:
import re

def chunk_text_sentences(text, chunk_chars=1200, overlap_chars=200):
    if not text or not text.strip():
        return []

    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    cur = ""

    for s in sents:
        s = s.strip()
        if not s:
            continue

        # si una sola frase ya supera el tamaño, la troceamos de forma defensiva
        if len(s) > chunk_chars:
            if cur:
                chunks.append(cur)
                cur = ""

            start = 0
            while start < len(s):
                end = start + chunk_chars
                piece = s[start:end]
                chunks.append(piece)
                start += max(1, chunk_chars - overlap_chars)
            continue

        if len(cur) + len(s) + 1 <= chunk_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = s

    if cur:
        chunks.append(cur)

    out = []
    for i, c in enumerate(chunks):
        if i == 0:
            out.append(c)
        else:
            prev = out[-1]
            overlap = prev[-overlap_chars:] if len(prev) > overlap_chars else prev
            out.append((overlap + " " + c).strip())

    return out


# reconstruir chunks con metadatos útiles para RAG
chunks = []

for d in docs:
    doc_text = d.get("text", "")
    doc_chunks = chunk_text_sentences(doc_text, chunk_chars=1200, overlap_chars=200)

    for i, c in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{d['id']}_chunk_{i}",
            "source_id": d.get("id"),
            "pmid": d.get("pmid"),
            "type": d.get("type"),
            "source_type_priority": 2 if d.get("type") == "meta-analysis" else 1,
            "title": d.get("title"),
            "chunk_index": i,
            "text": c
        })

print("N chunks:", len(chunks))
print("Chunk example:", chunks[0]["text"][:300] if chunks else "No chunks")

N chunks: 295
Chunk example: Can massage ameliorate menopausal and postmenopausal symptoms in women? A systematic review and meta-analysis. This systematic review and meta-analysis study explores the effect of massage on comprehensive menopause symptoms, including vasomotor, physical, psychological, sexual, sleep, anxiety, depr
N chunks: 295
Chunk example: Can massage ameliorate menopausal and postmenopausal symptoms in women? A systematic review and meta-analysis. This systematic review and meta-analysis study explores the effect of massage on comprehensive menopause symptoms, including vasomotor, physical, psychological, sexual, sleep, anxiety, depr


9) Embeddings + FAISS + retrieve()

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1) Modelo de embeddings
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2) Textos de los chunks
texts = [c["text"] for c in chunks]

# 3) Crear embeddings
emb = embed_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")

# 4) Crear índice FAISS con similitud coseno vía inner product
dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb)

print("Embeddings shape:", emb.shape)
print("FAISS index size:", index.ntotal)


# 5) Función de retrieval mejorada
def retrieve(query, k=10, initial_k=30, max_per_doc=1, use_type_priority=True):
    """
    Realiza la búsqueda en el corpus de documentos, ajustando el número de documentos recuperados
    y asegurando que no haya demasiados documentos del mismo tipo.
    """
    if not query or not query.strip():
        return []

    if len(chunks) == 0:
        return []

    initial_k = min(initial_k, len(chunks))
    k = min(k, len(chunks))

    q = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, idxs = index.search(q, initial_k)

    candidates = []
    for score, i in zip(scores[0], idxs[0]):
        ch = chunks[i]

        adjusted_score = float(score)
        if use_type_priority:
            adjusted_score += 0.02 * float(ch.get("source_type_priority", 1))

        candidates.append({
            "score": float(score),
            "adjusted_score": adjusted_score,
            "chunk_id": ch.get("chunk_id"),
            "source_id": ch.get("source_id"),
            "pmid": ch.get("pmid"),
            "type": ch.get("type"),
            "title": ch.get("title"),
            "chunk_index": ch.get("chunk_index"),
            "text": ch.get("text")
        })

    # Ordenar por score ajustado
    candidates = sorted(candidates, key=lambda x: x["adjusted_score"], reverse=True)

    # Limitar el número de documentos por fuente
    results = []
    seen_docs = {}

    for cand in candidates:
        doc_id = cand["source_id"]
        count = seen_docs.get(doc_id, 0)

        if count >= max_per_doc:
            continue

        results.append(cand)
        seen_docs[doc_id] = count + 1

        if len(results) >= k:
            break

    return results


# 6) Test retrieval
res = retrieve(
    "What effects does cognitive behavioral therapy have on menopausal insomnia?",
    k=8,
    initial_k=20,
    max_per_doc=1,
    use_type_priority=True
)

for r in res:
    print(
        f"score={r['score']:.3f} | adj={r['adjusted_score']:.3f} | "
        f"type={r['type']} | pmid={r['pmid']} | source={r['source_id']}"
    )
    print("title:", r["title"])
    print(r["text"][:220], "\n---\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embeddings shape: (295, 384)
FAISS index size: 295
score=0.757 | adj=0.797 | type=meta-analysis | pmid=41531400 | source=meta_41531400
title: Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis.
Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis. This study aimed to evaluate the effects of cognitive behavio 
---

score=0.766 | adj=0.786 | type=included-study-candidate | pmid=30481333 | source=included_30481333
title: Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene education.
Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene e

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embeddings shape: (295, 384)
FAISS index size: 295
score=0.757 | adj=0.797 | type=meta-analysis | pmid=41531400 | source=meta_41531400
title: Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis.
Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis. This study aimed to evaluate the effects of cognitive behavio 
---

score=0.766 | adj=0.786 | type=included-study-candidate | pmid=30481333 | source=included_30481333
title: Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene education.
Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene e

In [ ]:
# Pruebas pre-modelo

queries = [
    "How does CBT-I affect sleep quality in menopausal women?",
    "Does CBT-I improve insomnia severity during menopause?",
    "What evidence exists for CBT-I in postmenopausal women with vasomotor symptoms?"
]

for q in queries:
    print("\nQUERY:", q)
    res = retrieve(q, k=5, initial_k=15, max_per_doc=1, use_type_priority=True)
    for r in res:
        print(f"score={r['score']:.3f} | type={r['type']} | pmid={r['pmid']}")
        print("title:", r["title"])
        print("---")


QUERY: How does CBT-I affect sleep quality in menopausal women?
score=0.804 | type=included-study-candidate | pmid=33818845
title: The effect of group cognitive behavioural therapy for insomnia in postmenopausal women.
---
score=0.758 | type=meta-analysis | pmid=39145901
title: Prevalence of poor sleep quality during menopause: a meta-analysis.
---
score=0.747 | type=included-study-candidate | pmid=30481333
title: Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene education.
---
score=0.743 | type=included-study-candidate | pmid=26478725
title: Treatment of Insomnia, Insomnia Symptoms, and Obstructive Sleep Apnea During and After Menopause: Therapeutic Approaches.
---
score=0.721 | type=meta-analysis | pmid=41531400
title: Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review 

10) Generative model: Gemma

In [ ]:
from huggingface_hub import login
login()  # Esto abrirá un navegador donde te pedirá tus credenciales de Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "google/gemma-2-2b-it"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
)

model.to(device)
model.eval()

Device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

Device: cuda


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

11) Full RAG (build_prompt + answer_rag)

In [ ]:
import torch

def build_context(results, max_chars=3200, max_per_doc=1):
    """
    Construye el contexto para RAG asegurándose de que solo se incluyan fragmentos relevantes.
    """
    context_parts = []
    total_chars = 0
    seen_docs = {}

    for r in results:
        doc_id = r.get("source_id")

        count = seen_docs.get(doc_id, 0)
        if count >= max_per_doc:
            continue

        # Solo agregar textos relevantes que contengan la información más precisa
        piece = (
            f"[TITLE] {r.get('title', 'Unknown title')}\n"
            f"[PMID] {r.get('pmid', 'NA')}\n"
            f"[TYPE] {r.get('type', 'unknown')}\n"
            f"[TEXT] {r.get('text', '')}\n"
        )

        if total_chars + len(piece) > max_chars:
            break

        context_parts.append(piece)
        total_chars += len(piece)
        seen_docs[doc_id] = count + 1

    return "\n\n".join(context_parts)


def build_prompt(question, contexts):
    context_text = build_context(contexts, max_chars=3200, max_per_doc=1)

    prompt = f"""Eres un asistente médico especializado en salud hormonal femenina, ciclo menstrual, fertilidad, perimenopausia, menopausia, postmenopausia, salud tiroidea y cambios metabólicos.
    También eres experto en síntomas como alteraciones del sueño, estrés y otros cambios hormonales a lo largo de la vida de una mujer.

    Responde a la pregunta usando ÚNICAMENTE el contexto proporcionado.
    No inventes hechos.
    No añadas conclusiones generales que no estén presentes en el contexto.
    Si el contexto es insuficiente, responde exactamente: No hay suficiente información.

    Pregunta:
    {question}

    Contexto:
    {context_text}

    Respuesta:
    """
    return prompt

def expand_query_for_retrieval(question):
    q = question.lower()

    extra_terms = []

    if "cbt" in q or "cognitive behavioral" in q or "insomnia" in q:
        extra_terms.append(
            "CBT-I cognitive behavioral therapy insomnia menopausal women postmenopausal sleep quality insomnia severity ISI PSQI"
        )

    if "hot flash" in q or "hot flush" in q or "vasomotor" in q:
        extra_terms.append(
            "hot flashes hot flushes vasomotor symptoms menopause postmenopausal treatment intervention"
        )

    if "mental health" in q or "depression" in q or "anxiety" in q or "stress" in q:
        extra_terms.append(
            "menopause mental health depression anxiety stress psychological symptoms postmenopausal women"
        )

    if extra_terms:
        return question + " " + " ".join(extra_terms)

    return question

def generate_answer(prompt, max_new_tokens=180):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    )

    # Mover inputs al mismo dispositivo que el modelo
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            num_return_sequences=1
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return answer

def clean_answer(text):
    if not text:
        return ""

    # Limpiar las etiquetas de la conversación y líneas vacías
    text = text.replace("<|assistant|>", "").replace("<|user|>", "").replace("<|system|>", "").strip()
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    cleaned_lines = list(dict.fromkeys(lines))

    cleaned = "\n".join(cleaned_lines).strip()

    # Eliminar marcas de parada
    stop_markers = ["Question:", "Context:", "Answer:"]
    for marker in stop_markers:
        if marker in cleaned:
            cleaned = cleaned.split(marker)[0].strip()

    # Eliminar frases de conclusión genérica
    generic_tails = [
        "However, further research is needed",
        "Further research is needed",
        "More research is needed",
        "Additional research is needed",
        "This study concludes that"
    ]
    for tail in generic_tails:
        if tail in cleaned:
            cleaned = cleaned.split(tail)[0].strip()

    return cleaned


def answer_rag(question, k=5, initial_k=20, max_per_doc=2, max_new_tokens=180):
    retrieval_query = expand_query_for_retrieval(question)

    contexts = retrieve(
        retrieval_query,
        k=k,
        initial_k=initial_k,
        max_per_doc=max_per_doc,
        use_type_priority=True
    )

    context_text = build_context(contexts, max_chars=2800, max_per_doc=max_per_doc)

    prompt = f"""You are a biomedical research assistant.

    Use the provided context to answer the question.
    The context contains titles and abstracts from scientific articles.
    You may synthesize information across the provided studies.
    Do not invent facts that are not supported by the context.
    If the context is only partially sufficient, say what the context supports and what remains unclear.
    Only answer "Not enough information" if the context is completely unrelated to the question.

    Question:
    {question}

    Context:
    {context_text}

    Answer in English, in 2-4 clear sentences:
    """

    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    answer = clean_answer(answer)

    return {
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "prompt": prompt
    }


# Quick domain test

# Lista de preguntas de ejemplo
questions = [
    "Does CBT-I improve sleep quality and insomnia severity in menopausal women?",
    "How can I manage hot flashes during menopause?",
    "What are the effects of menopause on mental health?",
    "Is menopause the same for all women?",
    "Can stress contribute to menopause symptoms?",
    "How do hormonal changes affect sleep quality in postmenopausal women?",
    "What is the best treatment for menopausal insomnia?",
    "Does menopause increase the risk of heart disease?",
    "How can mindfulness help with menopausal symptoms?",
    "Is cognitive behavioral therapy effective for menopausal women?"
]

# Ejecutamos el modelo para cada pregunta
for q in questions[:3]:
    rag_result = answer_rag(
        q,
        k=3,
        initial_k=10,
        max_per_doc=1,
        max_new_tokens=120
    )

    print("\n" + "="*80)
    print("QUESTION:\n", rag_result["question"])
    print("\nANSWER:\n", rag_result["answer"])

    print("\nTOP CONTEXTS:")
    for c in rag_result["contexts"][:3]:
        print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")


QUESTION:
 Does CBT-I improve sleep quality and insomnia severity in menopausal women?

ANSWER:
 Based on the provided context, does CBT-I improve sleep quality and insomnia severity in menopausal women?
Yes, according to both the included study and the meta-analysis, CBT-I can significantly improve insomnia severity and sleep quality in menopausal women.

TOP CONTEXTS:
- PMID=33818845 | type=included-study-candidate | title=The effect of group cognitive behavioural therapy for insomnia in postmenopausal women.
- PMID=41531400 | type=meta-analysis | title=Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis.
- PMID=30481333 | type=included-study-candidate | title=Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene education.

QUESTION:
 How can I manage hot f

12) Baseline vs RAG comparison

In [ ]:
# =========================
# 12) Baseline vs RAG comparison
# =========================

def answer_base(question, max_new_tokens=220):
    prompt = f"""You are a helpful biomedical assistant specializing in female hormonal health,
    menstrual cycle, fertility, perimenopause, menopause, postmenopause, thyroid health, and metabolic changes.
    You are also an expert in symptoms such as sleep disturbances, stress, and other hormonal changes throughout a woman's life.

    Answer the question clearly in 2-4 sentences.
    If you are not sure, say so.

    Question:
    {question}

    Answer:
    """
    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    answer = clean_answer(answer)
    return answer

queries = {
    "Incertidumbre diagnóstica": [
        "¿Cómo sé si lo que me está pasando es menopausia o no?",
        "¿Esto que me pasa es por la edad o por otra cosa?",
        "¿Todo lo que me pasa ahora tiene que ver con la menopausia?",
        "¿Cómo diferencio si un síntoma es hormonal o no?",
        "¿Puede ser solo estrés y no menopausia?",
        "¿Estoy en menopausia aunque no tenga todos los síntomas?",
        "¿Esto es normal en esta etapa o debería preocuparme?",
        "¿Hay alguna forma de saber con certeza qué cosas son de la menopausia y cuáles no?",
        "¿Todo lo que me pasa ahora lo estoy relacionando mal con la menopausia?",
        "¿Estoy exagerando o realmente me está pasando algo?"
    ],

    "Necesidad de orientación personalizada": [
        "¿Qué pruebas médicas debería hacerme ahora mismo?",
        "¿Qué tipo de especialista debería consultar?",
        "¿Qué seguimiento debería llevar en esta etapa?",
        "¿Qué analíticas son importantes en esta etapa de la menopausia?",
        "¿Cómo sé si me falta alguna vitamina?",
        "¿Qué debería revisar en mis análisis de sangre?",
        "¿Qué controles debería hacerme de forma preventiva?",
        "¿Cuándo debería hacerme una densitometría?",
        "¿Qué revisiones son realmente necesarias y cuáles no?",
        "¿Cómo puedo saber si estoy bien controlada médicamente?"
    ],

    "Demanda de acompañamiento": [
        "¿Por qué nadie me explica claramente qué me está pasando?",
        "¿Existe alguna forma de tener seguimiento diario de mis síntomas?",
        "¿Cómo sería tener un acompañamiento diario en esta etapa?",
        "¿Qué información necesito para tomar decisiones sobre mi salud?",
        "¿Cómo puedo tener una guía clara de qué hacer en cada momento?",
        "¿Por qué siento que voy a ciegas en este proceso?",
        "¿Cómo puedo organizar toda la información que recibo?",
        "¿Cómo puedo saber qué es importante y qué no?",
        "¿Cómo puedo tener una visión global de mi estado?"
    ],

    "Prevención": [
        "¿Qué debería haber hecho antes para prevenir esto?",
        "¿A qué edad debería haber empezado a cuidarme?",
        "¿Qué puedo hacer ahora para evitar empeorar?",
        "¿Es tarde para empezar a prevenir?",
        "¿Qué hábitos son clave en esta etapa?",
        "¿Cómo puedo prevenir problemas futuros?",
        "¿Qué debería cambiar en mi estilo de vida?",
        "¿Qué cosas empeoran los síntomas sin darme cuenta?",
        "¿Cómo influye la alimentación en esta etapa?",
        "¿Qué debería haber sabido antes?"
    ],

    "Cuerpo y cambios físicos": [
        "¿Por qué me están pasando cosas en el cuerpo que nunca me habían pasado?",
        "¿Es normal que cambie la piel de repente?",
        "¿Por qué tengo picores o cambios en la piel?",
        "¿Es normal tener más sequedad corporal?",
        "¿Por qué mi cuerpo reacciona diferente ahora?",
        "¿Esto que me pasa en el cuerpo es temporal?",
        "¿Mi cuerpo va a volver a ser como antes?",
        "¿Estos cambios son reversibles?",
        "¿Cuánto tiempo duran estos cambios físicos?",
        "¿Qué cambios son normales y cuáles no?",
        "¿De verdad estos cambios físicos dependen tanto de la genética?",
        "¿Me voy a parecer más a mi madre y a mi abuela materna que a la familia de mi padre en cómo llevo la menopausia?",
        "¿Por qué me engorda la barriga más que todo lo demás?",
        "¿Me puedo quedar medio calva o sin pelo por la menopausia?",
        "¿Por qué parece que las famosas no tienen menopausia o se la han “curado”?"
    ],

    "Síntomas específicos": [
        "¿Qué puedo hacer con los sofocos?",
        "¿Hay algo que funcione de verdad para el calor?",
        "¿Por qué tengo tanto calor de repente?",
        "¿Los sofocos se van o se quedan para siempre?",
        "¿Por qué algunas de mis amigas casi no tienen sofocos y yo sí?",
        "¿Qué puedo hacer con el insomnio?",
        "¿Por qué estoy más cansada que antes?",
        "¿Por qué me enfermo más que antes?",
        "¿Es normal tener más infecciones ahora?",
        "¿Por qué tengo dolores nuevos que antes no tenía?",
        "¿Por qué siento que mi sistema inmunológico está peor?",
        "¿Estos síntomas son para siempre o en algún momento aflojan?"
    ],

    "Medicación y sistema sanitario": [
        "¿Por qué me recetan tantas cosas sin explicarme bien para qué son?",
        "¿Estoy tomando demasiados medicamentos?",
        "¿La menopausia se está medicalizando demasiado?",
        "¿Qué alternativas tengo a tomar medicación para todo?",
        "¿Cuándo es realmente necesaria la terapia hormonal?",
        "¿Por qué a unas mujeres se les receta terapia hormonal y a otras no?",
        "¿La medicación para la menopausia puede aumentar el riesgo de cáncer?",
        "Si como bien, ¿de verdad tengo que tomar suplementos?",
        "¿Todas las mujeres deberían tomar suplementos o tratamiento hormonal, salvo contraindicaciones?",
        "¿Tiene límite de tiempo el tratamiento hormonal sustitutivo?",
        "¿Es mejor tratar síntomas sueltos o ir a la causa?",
        "¿Cómo puedo evitar tomar medicamentos innecesarios?",
        "¿Qué decisiones médicas dependen realmente de mí y no solo del médico?"
    ],

    "Límites de la IA": [
        "¿Puede una app decirme qué tomar?",
        "¿Hasta dónde puede ayudar una IA en mi salud?",
        "¿Cómo uso una app sin sustituir al médico de toda la vida?",
        "¿Qué tipo de recomendaciones son seguras en una app?",
        "¿Cómo combinar la información de una app con la atención médica?",
        "¿Qué puede hacer una IA y qué no?",
        "¿Cómo saber cuándo tengo que ir a un profesional de la salud?",
        "¿Puede una app orientarme sin diagnosticarme?",
        "¿Cómo evitar engancharme demasiado a una app o a la IA?",
        "¿Cómo usar la tecnología para cuidarme mejor sin hacer locuras?"
    ],

    "Emoción y estado de ánimo": [
        "¿Cómo afecta la menopausia a mi estado emocional?",
        "¿Por qué estoy más irritable?",
        "¿Por qué me afectan más las cosas que antes?",
        "¿Esto emocional es hormonal o psicológico?",
        "¿Cómo influye el estrés en la menopausia?",
        "¿Por qué siento que no soy la misma?",
        "¿Cómo puedo gestionar mejor mis emociones?",
        "¿Esto es pasajero o se queda?",
        "¿Cómo recuperar algo de estabilidad emocional?",
        "¿Qué relación hay entre menopausia y ansiedad?",
        "¿Por qué estoy tan triste si, en teoría, estaba deseando no volver a tener la regla?",
        "¿Los síntomas de la menopausia se parecen a los de mi madre o mi abuela? Me da miedo acabar igual que ellas."
    ],

    "Sexo, deseo y relaciones": [
        "¿Por qué no tengo ganas de sexo si mi pareja me sigue gustando?",
        "¿A todas las mujeres se les baja la libido en esta etapa?",
        "¿Hasta qué punto la bajada de deseo tiene que ver con el miedo al deterioro físico o a dejar de ser atractiva?",
        "¿La sequedad vaginal es para siempre o se puede mejorar?",
        "¿Hay cosas que ya no se pueden hacer en la cama en esta etapa?",
        "Si cambian las hormonas y sube la testosterona, ¿eso puede cambiar lo que me atrae o de quién me enamoro?"
    ],

    "Niebla mental, memoria y cansancio cognitivo": [
        "¿Es normal que se me olviden palabras que tengo 'en la punta de la lengua' todo el rato?",
        "¿Por qué tengo la sensación de que estoy más lenta para pensar que antes?",
        "¿Esto de la 'niebla mental' es cosa de la menopausia o es que me estoy volviendo torpe?",
        "¿Cómo sé si estos despistes son normales o si deberían preocuparme?",
        "¿Por qué me cuesta más concentrarme en cosas que antes hacía sin pensar?",
        "¿Es normal acabar el día con la cabeza agotada aunque no haya hecho 'tanto'?",
        "¿Voy a recuperar mi agilidad mental o esto se queda así?",
        "¿La menopausia puede afectar a mi memoria a largo plazo o solo a estos despistes del día a día?",
        "¿Qué puedo hacer para mejorar la concentración y no sentir que tengo la cabeza 'nublada'?",
        "¿Cómo explico a los demás que no es que sea despistada, sino que estoy notando cambios mentales reales?"
    ]
}


for topic, topic_queries in queries.items():
    print(f"\nTopic: {topic}\n")
    for q in topic_queries:
        base = answer_base(q, max_new_tokens=220)
        rag_result = answer_rag(
            q,
            k=5,
            initial_k=15,
            max_per_doc=1,
            max_new_tokens=220
        )

        print("\n" + "=" * 80)
        print("QUESTION:", q)

        print("\nBASELINE ANSWER:")
        print(base)

        print("\nRAG ANSWER:")
        print(rag_result["answer"])

        print("\nTOP RAG SOURCES:")
        for c in rag_result["contexts"][:3]:
            print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")


Topic: Incertidumbre diagnóstica


QUESTION: ¿Cómo sé si lo que me está pasando es menopausia o no?

BASELINE ANSWER:
It's great that you're paying attention to your body! Menopause is a natural transition for women, but it can be confusing.  Here are some common signs of menopause: irregular periods, hot flashes, night sweats, vaginal dryness, mood swings, and difficulty sleeping. If you experience several of these symptoms consistently over time, it might be worth talking to your doctor about whether or not you're going through menopause.

RAG ANSWER:
To determine if you're experiencing menopause or not, look for changes in your menstrual cycle, such as irregular periods or a shorter length of time between periods. If these changes occur after age 40, it could be a sign of menopause.  However, it's important to consult a doctor for a proper diagnosis.

TOP RAG SOURCES:
- PMID=14412840 | type=included-study-candidate | title=Contemporary therapy of the menopausal syndrome.
- PMID=374

In [ ]:
# =========================
# Baseline vs RAG comparison 2
# =========================

def answer_base(question, max_new_tokens=220):
    prompt = f"""You are a helpful biomedical assistant specializing in female hormonal health,
    menstrual cycle, fertility, perimenopause, menopause, postmenopause, thyroid health, and metabolic changes.
    You are also an expert in symptoms such as sleep disturbances, stress, and other hormonal changes throughout a woman's life.

    Answer the question clearly in 2-4 sentences.
    If you are not sure, say so.

    Question:
    {question}

    Answer:
    """
    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    answer = clean_answer(answer)
    return answer

queries = [
    "Does CBT-I improve sleep quality and insomnia severity in menopausal women?",
    "How can I manage hot flashes during menopause?",
    "What are the effects of menopause on mental health?",
    "Is menopause the same for all women?",
    "Can stress contribute to menopause symptoms?",
    "How do hormonal changes affect sleep quality in postmenopausal women?",
    "What is the best treatment for menopausal insomnia?",
    "Does menopause increase the risk of heart disease?",
    "How do I differentiate if a symptom is hormonal or not?",
    "Can it be just stress and not menopause?"
]


for q in queries:
    base = answer_base(q, max_new_tokens=220)
    rag_result = answer_rag(
        q,
        k=5,
        initial_k=15,
        max_per_doc=1,
        max_new_tokens=220
    )

    print("\n" + "=" * 80)
    print("QUESTION:", q)

    print("\nBASELINE ANSWER:")
    print(base)

    print("\nRAG ANSWER:")
    print(rag_result["answer"])

    print("\nTOP RAG SOURCES:")
    for c in rag_result["contexts"][:3]:
        print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")


QUESTION: Does CBT-I improve sleep quality and insomnia severity in menopausal women?

BASELINE ANSWER:
Yes, there is evidence that Cognitive Behavioral Therapy for Insomnia (CBT-I) can be effective in improving sleep quality and reducing insomnia severity in menopausal women.  It helps them identify and change negative thoughts and behaviors related to sleep.

RAG ANSWER:
Based on the provided context, does CBT-I improve sleep quality and insomnia severity in menopausal women?
Yes, according to both the included study and the meta-analysis, CBT-I can significantly improve insomnia severity and sleep quality in menopausal women.

TOP RAG SOURCES:
- PMID=33818845 | type=included-study-candidate | title=The effect of group cognitive behavioural therapy for insomnia in postmenopausal women.
- PMID=41531400 | type=meta-analysis | title=Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analys